In [ ]:
import re, numpy as np, pandas as pd
from collections import Counter
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score

# Config
MAX_VOCAB, MAX_LEN, EMBED_DIM, BATCH, EPOCHS, LR = 10000, 100, 64, 64, 10, 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Data
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

# Vocab
def clean(t): return re.sub(r"[^a-z\s]", " ", str(t).lower()).split()

all_tokens = [t for tokens in train_df["customer_review"].apply(clean) for t in tokens]
vocab = {"<PAD>": 0, "<UNK>": 1}
for w, _ in Counter(all_tokens).most_common(MAX_VOCAB - 2):
    vocab[w] = len(vocab)

def encode(text):
    ids = [vocab.get(t, 1) for t in clean(text)[:MAX_LEN]]
    return ids + [0] * (MAX_LEN - len(ids))

# Tensors
X = torch.tensor([encode(t) for t in train_df["customer_review"]], dtype=torch.long)
y = torch.tensor(train_df["feedback"].values, dtype=torch.float)
X_test = torch.tensor([encode(t) for t in test_df["customer_review"]], dtype=torch.long)

split = int(0.85 * len(X))
train_loader = DataLoader(list(zip(X[:split], y[:split])), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(list(zip(X[split:], y[split:])), batch_size=BATCH)
test_loader  = DataLoader(X_test, batch_size=BATCH)

# Model
model = nn.Sequential(
    nn.Embedding(len(vocab), EMBED_DIM, padding_idx=0),
).to(DEVICE)

class TextClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(len(vocab), EMBED_DIM, padding_idx=0)
        self.fc    = nn.Sequential(nn.Linear(EMBED_DIM, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x):
        return self.fc(self.embed(x).mean(dim=1)).squeeze(1)

model     = TextClassifier().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()

# Train
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        criterion(model(xb.to(DEVICE)), yb.to(DEVICE)).backward()
        optimizer.step()

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            preds += (torch.sigmoid(model(xb.to(DEVICE))).cpu() >= 0.5).int().tolist()
            trues += yb.int().tolist()
    print(f"Epoch {epoch+1}  |  F1: {f1_score(trues, preds):.4f}")

# Predict
model.eval()
preds = []
with torch.no_grad():
    for xb in test_loader:
        preds += (torch.sigmoid(model(xb.to(DEVICE))).cpu() >= 0.5).int().tolist()

pd.DataFrame({"customer_review": test_df["customer_review"], "feedback": preds}).to_csv("submissions.csv", index=False)

Epoch 1  |  F1: 0.6127
Epoch 2  |  F1: 0.6127
Epoch 3  |  F1: 0.6127
Epoch 4  |  F1: 0.6127
Epoch 5  |  F1: 0.6127
Epoch 6  |  F1: 0.6127
Epoch 7  |  F1: 0.6127
Epoch 8  |  F1: 0.6127
Epoch 9  |  F1: 0.6543
Epoch 10  |  F1: 0.7114
